<a id="unit-u.317c6a6ae19e4388814ac18a04209e1a"></a>

<div>
  <div>08-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Көріністер және индекстермен сұрауларды жеделдету</div>
  <p>Сатып алу есебін қайта пайдаланып, қайталанатын агрегаттау мен байытуды алдын ала есептеңіз, содан кейін бір өнімді инверттелген индекспен зерттеңіз. Жоспар мен орындалу дәлелін түсіндірмес бұрын жауапты тексеріңіз.</p>
  <span>Көрініс · Синхронды материалданған көрініс · Асинхронды материалданған көрініс · Сұрауды қайта жазу · Инверттелген индекс · Query Profile</span>
</div>

**8-модуль: Apache Doris жүйесінде сұрауларды жеделдету** · [Сабақ](course.md) · [Тест](quiz08_acceleration_choices.ipynb)

Шамамен 55 минут жоспарлаңыз: Run All бірнеше минут алады, бірақ әр қадамда оқылатын бірнеше дәлел бар.

Сауда тобының екі қайталанатын қажеттілігі бар: түсім бақылау тақтасы және өнім деңгейіндегі зерттеу. Жеті қадаммен жұмыс істеңіз:

1. Бастапқы деректерді тексеріп, оқиға және өнім кестелерін дайындау.
2. Қалыпты көріністі көрсету: жауаптарды, таңдалған сканерлеулерді және нақты сканерлеу жұмысын салыстыру.
3. Жиынтықтарды бекітілген жазулармен сәйкес ұстау үшін синхронды материалданған көрініс пайдалану.
4. Көп кестелік санат есебін сақтап, жаңарту үшін асинхронды материалданған көрініс пайдалану.
5. Асинхронды материалданған көріністі қайта пайдалану үшін мөлдір сұрау қайта жазуын қолданып, кейін мағыналық шекарасын сынау.
6. Іріктемелі оқиға іздеуі үшін инверттелген индексті оқшаулау.
7. Таңдалған жоспардың артындағы статистика мен болжамдарды оқу.

**Жоғарыдан төменге орындаңыз.** Қалпына келтіру ұяшығы тек мұнда пайдаланылатын айқын аталған `m08_*` нысандарын қайта жасайды. Экспериментті қайталау немесе ішінара орындаудан кейін қалпына келтіру үшін бастапқы баптаудан қайта бастап, барлық ұяшықты орындаңыз. Бұл жұмыстың екі көшірмесін қатар орындамаңыз. Басқа модуль кестелері қолжетімді қалады.

<a id="unit-u.1852900af17241e88ec98d38fc443799"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі ұяшықты 1-бөлімге дейін орындаңыз. Ол ортақ курс көмекшісін жүктеп, кейінгі әр ұяшық пайдаланатын `lab` нысанын жасайды. Jupyter ядросын қайта іске қосқаннан кейін оны қайта орындаңыз. Ол Docker іске қоспайды және Doris деректерін өзгертпейді.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


<a id="unit-u.9d5ba90a1c0d4128956d2f983fde0c10"></a>

<a id="1-prepare-the-event-and-product-tables"></a>

## 1. Оқиға және өнім кестелерін дайындау

1-зертханалық жұмыстың бар жергілікті бір түйінді Doris 4.1.3 ортасын және `doris_course` ішіндегі мына кестелерді пайдаланыңыз:

| Қажетті кесте | Қайда дайындалған | Мұнда не үшін |
|---|---|---|
| `events_modelled` | 4-зертханалық жұмыс | Бір күндік типтелген егжей-тегжейлі оқиғалар |
| `dim_products` | 6-зертханалық жұмыс | Өнім санаттары мен брендтері |

Оқиға ауқымы — **[2020-03-01 00:00:00, 2020-03-02 00:00:00)**: басы кіреді, соңы кірмейді. Осы аралықта курс жиынында **75,259 оқиға** бар.

Келесі ұяшық сынақ ортасы тоқтап тұрса іске қосып, екі кестенің де толықтығын тексереді және аралықтағы оқиға санын көрсетеді. Кесте жоқ не толық емес болса, тоқтап, алдымен қай жұмысты аяқтау керектігін атайды. Сыртқы дерек қызметі немесе жүктеу қажет емес.

In [ ]:
from decimal import Decimal
from doris_course.ui import card

lab.start_container()
lab.connect()
lab.execute("USE doris_course")
lab.check_prerequisites(8)

lab.sql("""
SELECT COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-01' AND event_time < '2020-03-02'
""", title="Events in [2020-03-01, 2020-03-02)");

<a id="unit-u.ebb14ee325894c1b922def111fef2b9b"></a>

**Күтілетін нәтиже:** жасыл белгі `events_modelled` 10,158,080 жолмен және `dim_products` 204,231 жолмен барын көрсетеді; берілген аралықта `event_count` — **75,259**. Бұл тексерулер әр өріс мәнін емес, жол сандарын растайды.

<a id="prepare-the-lab-tables"></a>

### Зертханалық кестелерді дайындау

Есеп сатып алу оқиғаларын санап, `revenue` мәнін аймақ бойынша қосады. Түйіршіктілігі әр аймаққа бір жол; бірегей тұтынушыларды санамайды.

Келесі ұяшық тек осы жұмыс нысандарын қалпына келтіріп, тексерілген аралықтан оқиғалар көшіреді. `m08_events` Duplicate Key моделімен егжей-тегжейлі оқиғаларды сақтайды. Оның Key бағандары бірегейлікті емес, сұрыптау ретін анықтайды. `m08_products` Unique Key моделімен әр өнімге бір ағымдағы жол сақтайды. Тек көшірілген оқиғалар сілтейтін өнімдер қажет.

In [ ]:
lab.wait_for_mv_jobs("m08_events", "m08_sync_metrics", "m08_async_category_metrics")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS m08_async_category_metrics")
lab.execute("DROP VIEW IF EXISTS m08_purchase_view")
# Dropping m08_events also removes its own synchronous materialized index.
for table in ("m08_events_indexed", "m08_events_unindexed", "m08_events", "m08_products"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")

lab.execute("""
CREATE TABLE m08_events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO m08_events
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM events_modelled
WHERE event_time >= '2020-03-01' AND event_time < '2020-03-02'
""", title="Copy one day of event detail")
lab.execute("""
CREATE TABLE m08_products (
    product_id BIGINT NOT NULL,
    category VARCHAR(32),
    brand VARCHAR(32)
)
UNIQUE KEY(product_id)
DISTRIBUTED BY HASH(product_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO m08_products
SELECT product_id, category, brand
FROM dim_products
WHERE product_id IN (SELECT product_id FROM m08_events)
""", title="Copy the referenced products")
lab.execute("ANALYZE TABLE m08_events WITH SYNC")
lab.execute("ANALYZE TABLE m08_products WITH SYNC")
lab.sql("""
SELECT COUNT(*) AS unmatched_purchase_events
FROM m08_events e LEFT ANTI JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
""", title="Purchases without a matching product");

<a id="unit-u.c4a95ba8f6944073bc96c514f608c9e5"></a>

**Күтілетін нәтиже:** өзгермеген курс деректерінде `unmatched_purchase_events` — **501**. Бұлар 501 әртүрлі өнім емес, **`1005115`** өнімінің 501 сатып алу оқиғасы. 6-зертханалық жұмыс сәйкес емес байланысты көрсету үшін `dim_products` жасағанда осы өнімді әдейі алып тастаған (`WHERE product_id <> 1005115`). `m08_products` сол өлшемнен көшірілгендіктен, өнім мұнда да жоқ; бұл көшіру қатесі емес.

Аймақ есебі оқиға кестесін тікелей оқитындықтан бұл оқиғаларды қамтиды. Кейін санат есебі өнім кестесімен Inner Join қолданып, оларды алып тастайды, сондықтан жиынтықтары аймақ есебімен сәйкес болуы міндетті емес.

`ANALYZE ... WITH SYNC` осы өз кестелері үшін оңтайландырғыш статистикасын жинайды; материалданған нәтиже жасамайды.

<a id="unit-u.285b1a15d5e444b59e3d470801f06383"></a>

<a id="2-reuse-purchase-logic-with-a-regular-view"></a>

## 2. Қалыпты көрініспен сатып алу логикасын қайта пайдалану

Бұл қадам қалыпты көріністі көрсетеді: сұрау логикасын қалай қайта пайдаланады және есеп орындалғанда қандай жұмыс қалады. Бірнеше есепке бір ереже қажет: тек сатып алу оқиғаларын қосу. Қалыпты көрініс әр есеп `WHERE event_type = 'purchase'` қайталамасын деп сол ережені сақтайды.

Бұл қадам үш сұраққа бөлек жауап береді: жауаптар сәйкес пе, әр жоспар қай кестені оқиды және әр сканерлеу қанша жол өңдеді?

Әр сұраудың нақты сканерлеуі мен есептеуін байқау үшін жұмыс бұрынғы нәтижелерді немесе сүзу ақпаратын қайта пайдалана алатын үш кэшті уақытша өшіреді: SQL Cache, Query Cache және Condition Cache. Бұл кэш пайдасын көрініске немесе индекске телуге жол бермейді. Әр түсіруден кейін, түсіру сәтсіз болса да, бастапқы сеанс баптаулары қалпына келеді.

Көмекші құрал орындау егжей-тегжейлерін жинау үшін Profile level 2 де қосады. Сақтау, операциялық жүйе және инверттелген индекс кэштері орнында қалады, сондықтан өткен уақыт суық кэш сынағы емес.

Әр түсірілген нәтиженің астындағы жиналған панель түсіру орындаған нақты операторларды тізеді: `SET` операторлары, `EXPLAIN`, сұрау және сұраудың Profile ID мәнін қайтаратын `SELECT last_query_id()`. Панель сол Profile мәтінін FE HTTP API ішінен шығаратын `curl` пәрменімен аяқталады, сондықтан кез келген қадамды MySQL клиенті мен терминалдан қайталай аласыз.

<a id="unit-u.0d86ffd071f040c48e01670b1ddefcee"></a>

**A. Жауаптарды салыстыру.** Қалыпты көріністі жасап, аймақ есебін тікелей `m08_events` кестесінде және `m08_purchase_view` арқылы орындаңыз. Көріністе сатып алу сүзгісі бар, сондықтан `view_query` оны қайталамайды.

Төмендегі екі шағын нәтиже кестесін оқыңыз: әр жол — бір аймақ, `event_count` сатып алу оқиғаларын санайды, ал `total_revenue` олардың түсімін қосады. Тексеру әр топ пен метриканы салыстырады.

In [ ]:
lab.execute("DROP VIEW IF EXISTS m08_purchase_view")
lab.execute("""
CREATE VIEW m08_purchase_view AS
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM m08_events
WHERE event_type = 'purchase'
""")
regional_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m08_events
WHERE event_type = 'purchase'
GROUP BY region
ORDER BY region
"""
view_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m08_purchase_view
GROUP BY region
ORDER BY region
"""
baseline = lab.capture_query(regional_query)
regular_view = lab.capture_query(view_query)
baseline.show_result("Direct query: purchases by region")
regular_view.show_result("Through the view: purchases by region")
lab.assert_same_rows(baseline.rows, regular_view.rows)
card("All 8 regional groups have the same purchase counts and revenue.",
     "ok", "Answers match");

<a id="unit-u.2dbd984b57a4454ab52377fcb641c1ff"></a>

**Күтілетін нәтиже:** екі кестеде сол сегіз аймақ тобы бар. Мысалы, өзгермеген курс деректерінде `region_01` 2,319 сатып алу оқиғасына және 732,977.08 түсімге ие. Бұл көріністі пайдалану есеп жауабын сақтайтынын тексереді. Дерекқор аз жұмыс істеді ме, оны әлі көрсетпейді.

<a id="unit-u.4bb533c7d2f8460ebd59147602a50b0a"></a>

**B. Әр жоспар не оқитынын тексеру.** Келесі ұяшық түсірілген `EXPLAIN` жоспарларынан тек таңдалған `TABLE:` жолдарын көрсетеді. `m08_events(m08_events)` ішінде жақша алдындағы атау — кесте, ішіндегі — оның таңдалған негізгі индексі. Толық жоспарлар жиналған панельдерде бар.

Бұл ұяшық A бөлігінде түсірілген дәлелді көрсетеді; есептерді қайта орындамайды.

In [ ]:
baseline.show_scan_plan("Direct query: selected scan")
regular_view.show_scan_plan("Through the view: selected scan")
lab.assert_scan(baseline, "m08_events")
lab.assert_scan(regular_view, "m08_events");

<a id="unit-u.ffbca1f09d0544188ad3737e218a8dbf"></a>

**Күтілетін нәтиже:** таңдалған екі Scan жолы да `m08_events(m08_events)` атайды. Екінші SQL қалыпты көріністі атаса да, жоспары астындағы оқиға кестесін әлі оқиды. Қалыпты көрініс бөлек тұрақты есеп нәтижесін емес, сұрау анықтамасын сақтайды. Жоспар қатынау жолын көрсетеді; нақты сканерленген жолдарды өлшемейді.

<a id="unit-u.153bcd861535495e97665bee6f210890"></a>

**C. Нақты сканерлеу жұмысын салыстыру.** Төмендегі әр `OLAP_SCAN_OPERATOR` астынан `ScanRows` және `RowsProduced` оқыңыз. Бұлар Fragment, Pipeline және кесте/индекс сәйкестігі сақталған біріктірілген Query Profile өңделмеген санағыштары. Doris жоспардың әр бөлігін бір немесе бірнеше орындау данасы ретінде орындайды: әрқайсысы деректің бір бөлігін оқитын параллель көшірмелер; оларды 9-модуль (Apache Doris жүйесінде шардтау, репликация және үлестірілген орындау) сипаттайды. Әр санағыш алдымен даналар бойынша `sum`, кейін `avg`, `max` және `min` береді. Бұл салыстыруға `sum` пайдаланыңыз; қалған үшеуін оған ешқашан қоспаңыз.

- `ScanRows`: осы сканерлеу сақтау орнынан оқыған жолдар, бет кэшінен оқуды қоса.
- `RowsProduced`: сканерлеу жағындағы сүзуден кейін осы сканерлеу кейінгі операторларға жіберетін жолдар. Бұл соңғы есеп жолдарының саны емес.

Толық өңделмеген Profile жазбалары әрі қарай тексеруге жиналған панельдерде қолжетімді қалады.

In [ ]:
baseline.show_scan_rows("Direct query: actual scan rows")
regular_view.show_scan_rows("Through the view: actual scan rows");

<a id="unit-u.16899c811e7e45f6b6b4e0e8f366e396"></a>

**Күтілетін нәтиже:** сыналған деректер жиынында екі сканерлеу де `ScanRows: sum 75.259K (75259)` және `RowsProduced: sum 18.298K (18298)` көрсетеді:

```text
75,259 event rows scanned
    → 18,298 purchase events emitted by the scan
    → 8 regional groups produced by the later aggregation
```

Әр жиналған өңделмеген Profile үстіндегі біріктірілген Profile ішінде екі санағыш Scan операторы астында тұр. Fragment және Pipeline нөмірлері мен `nereids_id` сіздің орындауда өзгеше болуы мүмкін. Сканерлеудің орындау даналары санына сай `instance_num` және `avg`, `max`, `min` мәндері де өзгереді; бір дана болса, төрт мәннің бәрі `sum` мәніне тең:

```text
Fragment 2:
  Pipeline 0(instance_num=4):
    OLAP_SCAN_OPERATOR(nereids_id=189. table_name=m08_events(m08_events))(id=0):
       - RowsProduced: sum 18.298K (18298), avg 4.574K (4574), max 4.748K (4748), min 4.445K (4445)
       - ScanRows: sum 75.259K (75259), avg 18.814K (18814), max 19.145K (19145), min 18.482K (18482)
```

Қалыпты көрініс сүзгі анықтамасын қайта пайдаланды, бірақ осы орындаулардағы тіркелген сканерлеу жұмысын азайтпады. Тең сұрау нәтижелері, таңдалған негізгі кесте сканерлеулері және орындалу жол сандары үш бөлек бақылауды қолдайды. Тұрақты уақыт қатынасын анықтамайды.

Келесі бөлім сол есеп SQL сақтап, синхронды материалданған көрініс оның орнына алдын ала есептелген агрегаттар бере ала ма, тексереді.

<a id="unit-u.6dc8a90a0ee44216a497a0cf5ac1bd3f"></a>

<a id="3-keep-summaries-current-with-a-synchronous-materialized-view"></a>

## 3. Синхронды материалданған көрініспен жиынтықтарды өзекті ұстау

Қалыпты көрініс сүзгі анықтамасын қайта пайдаланды, бірақ екі сұрау да егжей-тегжейлі оқиғаларды сканерледі. Бақылау тақтасына сол сан мен түсім қайта-қайта қажет, сондықтан енді **синхронды материалданған көрініспен** метрикаларды алдын ала есептеңіз.

Бұл қадам төрт нәрсені бөлек тексереді: алғашқы құру аяқталады, есеп жауабы бірдей қалады, жоспар материалданған индексті таңдайды және нақты сканерлеу жұмысы азаяды. Кейін жазу сәйкестігін сынау үшін бір сатып алу қосады.

<a id="unit-u.fdbde36b20824f2fa14cafb4a4f4dbf3"></a>

<a id="a-create-the-summary-and-wait-for-the-initial-build"></a>

### A. Жиынтықты жасап, алғашқы құрылуын күту

Анықтама оқиғаларды `(region, event_type)` бойынша топтап, әр топқа оқиға саны мен түсім қосындысын сақтайды. `event_type` сақтау сатып алу есебіне басқа түрлерді алып тастауға мүмкіндік береді. `region` сақтау бұрынғы аймақ жиынтықтарын береді. Мысалы, `region_01` ішіндегі сатып алу мен себет оқиғалары бөлек топтарға тиесілі.

Сыналған құру талап еткендей, `mv_` бүркеншік атаулары сақталған бағандарға негізгі кесте бағандарынан бөлек атаулар береді. Көрініс өз Duplicate Key оқиға кестесінде жасалады; бұл мысал әр сұрау немесе кесте моделі сол анықтаманы қолдайды дегенді білдірмейді.

Алғашқы құру фондық тапсырма ретінде жүреді. `sync_before` ескі Job ID мәндерін тіркейді, сондықтан күту бұрынғы сәттілікті емес, осы жаңа құруды тексереді. Ықшам нәтижеден **Build state** оқыңыз; толық тапсырма жазбасы және күту қайталайтын `SHOW ALTER TABLE MATERIALIZED VIEW` операторы жиналған панельдерде қолжетімді.

Содан кейін `ANALYZE TABLE ... WITH SYNC` осы кестеге оңтайландырғыш статистикасын жинайды. Мұнда `WITH SYNC` статистика жинаудың аяқталуын күту дегенді білдіреді; материалданған көріністің жазумен қолдау әрекетінен бөлек.

In [ ]:
lab.wait_for_mv_jobs("m08_events", "m08_sync_metrics", "m08_async_category_metrics")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS m08_sync_metrics ON m08_events")
sync_before = {str(row["JobId"]) for row in lab.sync_mv_jobs("m08_events", "m08_sync_metrics")}
lab.execute("""
CREATE MATERIALIZED VIEW m08_sync_metrics AS
SELECT region AS mv_region,
       event_type AS mv_event_type,
       COUNT(*) AS mv_event_count,
       SUM(revenue) AS mv_total_revenue
FROM m08_events
GROUP BY region, event_type
""")
lab.wait_for_sync_mv("m08_events", "m08_sync_metrics", sync_before)
lab.execute("ANALYZE TABLE m08_events WITH SYNC");

<a id="unit-u.29d3825207ff487090806cd23fa1b5b6"></a>

**Күтілетін нәтиже:** нәтиже негізгі `m08_events` кестесін, `m08_sync_metrics` материалданған көрінісін және **`FINISHED`** құру күйін атайды. Алғашқы құрылым дайын. Бұл есеп сұрауы оны таңдайтынын әлі дәлелдемейді. Құру сәтсіз болса немесе шектелген күту ішінде аяқталмаса, көмекші құрал қате береді.

<a id="unit-u.d07f2dc446cb4708af578a27b86d5a5e"></a>

<a id="b-run-the-original-report-and-check-its-answer"></a>

### B. Бастапқы есепті орындап, жауабын тексеру

2-қадамдағы сол `regional_query` орындаңыз. Ол әлі `m08_events` атап, сатып алу оқиғаларын сүзеді; синхронды көріністен таңдайтындай өзгертілмеген.

`capture_query` есепті бір рет орындап, нәтижесін, жоспарын және Query Profile сақтайды. Бұл ұяшық тек сегіз жолдық нәтижені көрсетеді. `assert_same_rows` әр аймақ пен екі метриканы сақталған материалдандыруға дейінгі бастапқы нұсқамен салыстырады; тек жалпы жиынтықтарды салыстырмайды.

In [ ]:
sync_report = lab.capture_query(regional_query)
sync_report.show_result("Regional purchases after synchronous materialization")
lab.assert_same_rows(baseline.rows, sync_report.rows)
card("Every region has the same purchase count and revenue as the Step 2 baseline.",
     "ok", "Report answer unchanged");

<a id="unit-u.6b42934877c7403cabf052da4af5a979"></a>

**Күтілетін нәтиже:** есепте әлі сегіз аймақ тобы бар. Мысалы, `region_01` әлі 2,319 сатып алу оқиғасына және 732,977.08 түсімге ие. Тексеру жауаптың өзгермегенін растайды. Жалғыз нәтиже теңдігі оны қай сақтау құрылымы бергенін немесе қанша жұмыс істелгенін көрсетпейді.

<a id="unit-u.0c4f4b9bd83a48fcb3ff4bfd51bd1913"></a>

<a id="c-confirm-that-the-plan-selected-the-materialized-index"></a>

### C. Жоспар материалданған индексті таңдағанын растау

Жасауға дейінгі және кейінгі таңдалған Scan жолдарын салыстырыңыз. Жақша алдындағы атау — негізгі кесте; жақша ішіндегі — таңдалған индекс. Әр үзінді астында толық жоспарлар жиналған.

Бұл — есептің екі орындалуымен бұрын түсірілген жоспарлар. Оларды көрсету сұрауларды қайта орындамайды.

In [ ]:
baseline.show_scan_plan("Before creation: selected scan")
sync_report.show_scan_plan("After creation: selected scan")
lab.assert_scan(sync_report, "m08_events", "m08_sync_metrics");

<a id="unit-u.8ac764000677473fa64c2fdda3b4e13b"></a>

**Күтілетін нәтиже:** таңдалған қатынау жолы мынадан:

```text
m08_events(m08_events)
```

мынаған өзгереді:

```text
m08_events(m08_sync_metrics)
```

SQL әлі негізгі кестеге сілтейді, бірақ жаңа жоспар синхронды материалданған индексті таңдайды. Бұл жұмыс синхронды көріністі осылай қолданады; оған бөлек кесте ретінде сұрау жасалмайды. Бұл нақты сканерленген жол санын емес, жоспар таңдауын дәлелдейді. Күтілетін Scan жоқ болса, көрініс қолданылды деп мәлімдемес бұрын толық жоспарды тексеріңіз.

<a id="unit-u.0b963865b5594764a1dbb6537c238da1"></a>

<a id="d-compare-the-actual-scan-work-recorded-in-query-profile"></a>

### D. Query Profile тіркеген нақты сканерлеу жұмысын салыстыру

`baseline` және `sync_report` бұрын `lab.capture_query()` арқылы жасалды. Ол көмекші Profile қосты, әр сұрауды орындады және Query Profile жазбасын нәтиже мен жоспармен бірге сақтады. Төмендегі шақырулар SQL қайта орындамайды: `show_profile_scan_rows()` әр сақталған Profile ішінен `MergedProfile` оқып, аталған Scan операторы астынан тек **`ScanRows`** және **`RowsProduced`** алады.

`ScanRows` сканерлеу оқыған жолдарды, бет кэшін қоса тіркейді. `RowsProduced` сол сканерлеу кейінгі операторларға шығарған жолдарды тіркейді. Ешқайсысы автоматты түрде соңғы есеп жолдарының саны емес. Кесте/индекс белгілері әр орындалу санағышын C бөліміндегі `EXPLAIN` анықтаған қатынау жолымен байланыстырады; толық өңделмеген Profile жазбалары қосымша қарауға жиналған күйінде қалады.


In [ ]:
baseline.show_profile_scan_rows("Before creation: base-table scan work")
sync_report.show_profile_scan_rows("After creation: materialized-index scan work");

<a id="unit-u.9d2c8d1454ed457ea8d008df04862a5e"></a>

**Күтілетін нәтиже:** бастапқы нұсқа 75,259 сканерленген оқиға жолын тіркеп, 18,298 сатып алу оқиғасын шығарады. Тексерілген орындауда материалданған индекс сканерлеуі сақталған 64 жиынтық жолын оқып, 32 сатып алу-жиынтық жолын шығарады. Екі есеп те сол сегіз аймақ тобын қайтарады.

```text
Base-table path:
75,259 event rows scanned → 18,298 purchase events → 8 regional results

Observed materialized-index path:
64 summary rows scanned → 32 purchase-summary rows → 8 regional results
```

Екі жол әртүрлі түйіршіктіліктегі жолдарды оқиды: бастапқы оқиғалар және алдын ала есептелген жиынтықтар. Ішінара топтар бірнеше Tablet ішінде сақталып, кейінгі агрегаттаумен біріктірілуі тиіс, сондықтан 64 жиынтық жолын сканерлеу есепте 64 жол болуы тиіс дегенді білдірмейді. Нақты жиынтық жол сандары сақтау күйімен өзгеруі мүмкін; бекітілген 64 немесе 32 талап етпей, өз орындауыңыздағы санағыштарды пайдаланыңыз.

Тексерулер бірге бұл орындаудың материалданған индексті таңдағанын, сол жауапты қайтарғанын және аз жол сканерлегенін көрсетеді. Тұрақты кідіріс жеделдеуін анықтамайды және қосымша сақтау мен жазумен қолдау шығынын өлшемейді. Келесі эксперимент бекітілген жазудан кейін жауаптың да дұрыс қалатынын тексереді.

<a id="unit-u.1c4bd10b0fa04891997dd0682fc2d78e"></a>

<a id="e-verify-synchronous-maintenance-after-one-committed-purchase"></a>

### E. Бір бекітілген сатып алудан кейін синхронды қолдауды тексеру

`region_01` ішіне `19.95` құнды бір сатып алу қосып, жаңарту бермей өзгермеген аймақ есебін орындаңыз. Бұрынғы тексерулер толық бастапқы нұсқаны анықтады; бұл нысаналы тексеру тек жаңа жол әсер еткен топты және таңдалған сканерлеуді көрсетіп, тексереді.


In [ ]:
lab.insert("""
INSERT INTO m08_events
SELECT CAST('2020-03-01 12:00:00' AS DATETIME), -8001, -8001,
       'purchase', 'region_01',
       (
           SELECT MIN(p.product_id)
           FROM m08_products p
           JOIN m08_events e ON p.product_id = e.product_id
           WHERE e.event_type = 'purchase'
       ),
       CAST(19.95 AS DECIMAL(12,2))
WHERE NOT EXISTS (SELECT 1 FROM m08_events WHERE event_id = -8001)
""", title="Append one owned purchase once");

<a id="unit-u.d91b6bfcfd8e4d8f8d3df54abb73e45c"></a>

Толық есеп SQL төменде көрінеді. `verify_query_change()` `SELECT` орындап, `region_01` өзгерістерін тексереді. Бөлек `EXPLAIN` шақыруы жоспар дәлелін айқын етіп, тек таңдалған `TABLE` жолын көрсетеді.


In [ ]:
after_insert_sql = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m08_events
WHERE event_type = 'purchase'
GROUP BY region
ORDER BY region
"""

after_insert_rows = lab.verify_query_change(
    after_insert_sql,
    baseline.rows,
    match={"region": "region_01"},
    expected_changes={"event_count": 1, "total_revenue": Decimal("19.95")},
    title="Observed change in region_01",
)
lab.explain_selected_scans(
    "EXPLAIN " + after_insert_sql,
    title="After the write: selected scan",
    expected_table="m08_events",
    expected_index="m08_sync_metrics",
);


<a id="unit-u.a1e9e5e5fa6e4998871c3cf303bfa0a6"></a>

**Күтілетін нәтиже:**

- Нәтиже үзіндісі `region_01` үшін `event_count_change = 1` және `total_revenue_change = 19.95` көрсетеді. Бұл мәндер бекітілген сатып алудың есеп нәтижесінде барын дәлелдейді.
- Қысқартылған `EXPLAIN` үзіндісі `TABLE: doris_course.m08_events(m08_sync_metrics)` көрсетеді. Бұл жоспардың синхронды материалданған индексті таңдағанын дәлелдейді.

Енгізу мен есептің арасында қолмен жаңарту жоқ. Нәтиже және жоспар үзінділері әртүрлі сұраққа жауап береді: нәтиже жаңарған жауапты тексереді, ал `EXPLAIN` жоспарланған қатынау жолын анықтайды. Толық есеп, жоспар және Query Profile көрсетілмейді, өйткені бұл нысаналы тексеруге жаңа дәлел қоспайды.


<a id="unit-u.d2ccd3f38bae4fa5af6681cc3504f50e"></a>

<a id="4-store-and-refresh-a-multi-table-report-with-an-asynchronous-materialized-view"></a>

## 4. Көп кестелік есепті асинхронды материалданған көрініспен сақтау және жаңарту

Келесі есеп сатып алуларды санат пен аймақ бойынша топтайды. Оған ағымдағы өнім атрибуттарымен Join қажет, оны бір кестелік жиынтық бере алмайды. Асинхронды материалданған көрініс осы көп кестелік нәтижені сақтайды және тікелей сұрала алады.

Анықтаманы алғашқы жаңартудан бөлу үшін `BUILD DEFERRED`, жаңарту шартын айқын ету үшін `REFRESH COMPLETE ON MANUAL` және мөлдір қайта жазуға ескірген нәтиже жеңілдігін қоспау үшін `grace_period = 0` пайдаланыңыз. Сақталған көріністі тікелей оқу және оңтайландырғыштан негізгі кесте сұрауын қайта жазуды сұрау — бөлек амалдар.

In [ ]:
lab.wait_for_mv_jobs("m08_events", "m08_sync_metrics", "m08_async_category_metrics")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS m08_async_category_metrics")

category_query = """
SELECT p.category, e.region,
       COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category, e.region
ORDER BY p.category, e.region
"""
stored_category_query = """
SELECT category, region, event_count, total_revenue
FROM m08_async_category_metrics
ORDER BY category, region
"""
lab.execute("""
CREATE MATERIALIZED VIEW m08_async_category_metrics
BUILD DEFERRED REFRESH COMPLETE ON MANUAL
DISTRIBUTED BY HASH(category) BUCKETS 4
PROPERTIES ("replication_num" = "1", "grace_period" = "0")
AS
SELECT p.category, e.region,
       COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category, e.region
""")
refresh_before = {str(row["TaskId"]) for row in lab.async_mv_tasks("m08_async_category_metrics")}
lab.execute("REFRESH MATERIALIZED VIEW m08_async_category_metrics COMPLETE")
lab.wait_for_async_refresh("m08_async_category_metrics", refresh_before)
lab.sql("""
SELECT Name, State, RefreshState, RefreshInfo, SyncWithBaseTables
FROM mv_infos('database'='doris_course')
WHERE Name = 'm08_async_category_metrics'
""", title="Asynchronous view status")
with lab.session_settings({"enable_materialized_view_rewrite": "false"}):
    category_before, stored_before = lab.compare_queries(
        category_query, stored_category_query,
        left_label="source tables", right_label="stored view",
        title="First refresh: complete result comparison",
    )


<a id="unit-u.1765d6515e9844f590a7bb0b507ac7aa"></a>

**Нәтижені қалай оқу керек:**

1. **Аяқталған қолмен жаңарту тапсырмасы:** жаңа Task ID `SUCCESS` күйіне жетіп, `COMPLETE` жаңарту пайдаланады және 100% ілгерілеу көрсетеді. Көмекші құрал жаңадан жіберілген тапсырманы күтеді, сондықтан ескі сәтті тапсырма тексеруге сай келе алмайды. Жиналған панель қайталанатын `tasks("type"="mv")` сұрауын көрсетеді.
2. **Асинхронды көрініс күйі:** айқын SQL проекциясы тек нысан атауын, нысан күйін, соңғы жаңарту күйін, жарияланған жаңарту саясатын және сақталған нәтиженің негізгі кестелермен синхрондалғанын көрсетеді.
3. **Толық нәтиже салыстыруы:** екі SQL операторы да 64 санат/аймақ тобын және `differing_rows = 0` қайтарады. Көмекші құрал екі 64 жолдық кестені көрсетпей, әр баған мен қайталану еселігін салыстырады. Бастапқы сұрауда материалданған көрініс қайта жазуы өшірілген, сондықтан сақталған көріністі өзімен кездейсоқ салыстырмайды.

Ноутбук көрінетін `category_query` және `stored_category_query` SQL жолдарын салыстыру көмекшісіне әлі береді. Қысқартылған кесте тек көрсетілім қорытындысы; теңдік тексеруі толық сұрау нәтижелерін әлі пайдаланады.

<a id="unit-u.e42cc6bd22204dc6b2b17b6ce937b7d7"></a>

<a id="change-one-products-category-and-inspect-the-stale-stored-result"></a>

### Бір өнім санатын өзгертіп, ескірген сақталған нәтижені тексеру

Сатып алу оқиғалары бар өнімді таңдап, өз өлшеміңізде оны `lab8_reclassified` санатына көшіріңіз. Бұл жалпы оқиға саны мен түсімді өзгертпей, бар сатып алулары қай санатқа тиесілі болуы керектігін өзгертеді.

Келесі жаңартуды сұрамас бұрын сақталған көріністі оқыңыз. Қалыпты қайта жазу қосылған негізгі кесте жоспарын да көрсетіңіз; тікелей ескірген оқу мен мөлдір қайта жазу бірдей әрекет етеді деп ойламай, нақты көрген жоспарды түсіндіріңіз.

In [ ]:
chosen_product = lab.sql("""
SELECT p.product_id, p.category, COUNT(*) AS purchase_events
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.product_id, p.category
ORDER BY purchase_events DESC, p.product_id
LIMIT 1
""", title="Product to reclassify")
changed_product_id = int(chosen_product.iloc[0]["product_id"])
original_category = str(chosen_product.iloc[0]["category"])
assert original_category != "lab8_reclassified", "Restart from Section 1 to repeat this experiment."
lab.execute(f"""
UPDATE m08_products SET category = 'lab8_reclassified'
WHERE product_id = {changed_product_id}
""")
with lab.session_settings({"enable_materialized_view_rewrite": "false"}):
    category_changed, stored_stale = lab.summarize_query_values(
        category_query, stored_category_query,
        left_label="current source", right_label="stored view before refresh",
        value_column="category",
        values=(original_category, "lab8_reclassified"),
        metrics=("event_count", "total_revenue"),
        title="Old and new category before refresh",
    )
lab.assert_same_rows(stored_before, stored_stale)
assert "lab8_reclassified" in set(category_changed["category"])
assert "lab8_reclassified" not in set(stored_stale["category"])
assert category_changed["event_count"].sum() == category_before["event_count"].sum()
assert category_changed["total_revenue"].sum() == category_before["total_revenue"].sum()
with lab.session_settings({"enable_materialized_view_rewrite": "true"}):
    lab.explain_selected_scans("EXPLAIN " + category_query,
                               title="Selected scans while the stored view is stale")
card("The stored view is unchanged, while the source answer moves purchases into lab8_reclassified.",
     "ok", "Refresh lag observed");

<a id="unit-u.ea136b1c7e0d4516911e99cfffd83da6"></a>

**Нәтижені қалай оқу керек:**

Екі `result` жолының арасындағы сол санатты салыстырыңыз:

- **current source** ішінде 508 сатып алу және `122846.80` түсім `category_8` ішінен `lab8_reclassified` ішіне ауысты.
- **stored view before refresh** ішінде `category_8` сол сатып алуларды әлі қамтиды, ал `lab8_reclassified` ішінде нөл топ, оқиға және түсім бар.
- Үнсіз тексерулер жалпы оқиға саны мен түсімнің өзгермегенін де растайды. Бұл енгізу не жою емес, қайта жіктеу.
- Қысқартылған `EXPLAIN` сақталған көрініс ескіргенде таңдалған бастапқы кесте сканерлеулерін тізеді. Жоспарды сипаттайды; көріністі жаңартпайды және орындалу жұмысын өлшемейді.

Көмекші құрал толық `category_query` және `stored_category_query` SQL жолдарын алады. Екі толық нәтижені орындайды, бірақ ескірген көшірмені түсінуге қажетті тек бастапқы және жаңа санаттарды көрсетеді.


In [ ]:
refresh_before = {str(row["TaskId"]) for row in lab.async_mv_tasks("m08_async_category_metrics")}
lab.execute("REFRESH MATERIALIZED VIEW m08_async_category_metrics COMPLETE")
lab.wait_for_async_refresh("m08_async_category_metrics", refresh_before)
with lab.session_settings({"enable_materialized_view_rewrite": "false"}):
    category_after_refresh, stored_fresh = lab.summarize_query_values(
        category_query, stored_category_query,
        left_label="current source", right_label="refreshed stored view",
        value_column="category",
        values=(original_category, "lab8_reclassified"),
        metrics=("event_count", "total_revenue"),
        title="Old and new category after refresh",
    )
lab.assert_same_rows(category_changed, category_after_refresh)
lab.assert_same_rows(category_after_refresh, stored_fresh)
lab.execute("ANALYZE TABLE m08_products WITH SYNC")
lab.execute("ANALYZE TABLE m08_async_category_metrics WITH SYNC")
card("The refreshed category/region groups now match the changed source answer.",
     "ok", "Refresh reconciliation passed");

<a id="unit-u.569215086907405e96cb29121326082c"></a>

**Күтілетін нәтиже:**

- Жаңа жаңарту Task ID `SUCCESS` күйіне жетіп, `COMPLETE` жаңарту пайдаланады және 100% ілгерілеу көрсетеді.
- Ағымдағы дереккөзде де, жаңарған сақталған көріністе де `category_8` енді 1,814 оқиға және `551991.97` түсім, ал `lab8_reclassified` 508 оқиға және `122846.80` түсім қамтиды.

Тексерулер барлық 72 топ бойынша әр баған мен қайталану еселігін әлі салыстырады. Материалданған көріністі өзімен салыстырмау үшін бастапқы сұрауда қайта жазу өшірілген. Толық жаңарту сақталған сұрауды қайта есептейді; бұл жұмыс өндірістік ресурс шығынын өлшемейді.


<a id="unit-u.cb274a169c934a0cb8542d5e9744786f"></a>

<a id="5-reuse-an-asynchronous-materialized-view-through-transparent-query-rewrite"></a>

## 5. Асинхронды материалданған көріністі мөлдір сұрау қайта жазуы арқылы қайта пайдалану

Қолданба негізгі кестелерге сұрауды жалғастыра алады; `m08_async_category_metrics` атауы қажет емес. Мөлдір қайта жазу қосылса, көріністе сұралған жауапты шығаруға жеткілікті ақпарат болғанда Doris жаңартылған көріністі алмастыра алады.

Бұл көрініс үшін төмендегі жарамды сұрау:

- сол `m08_events`–`m08_products` Join және сол `event_type = 'purchase'` ауқымын пайдаланады;
- `category` бойынша топтайды, оны Doris көріністің ұсақтау `category, region` топтарын ірілендірумен ала алады;
- тек көрініс сақтаған қосылатын метрикалар `COUNT(*)` және `SUM(revenue)` сұрайды.

Көріністе пайдалануға болатын жаңартылған нәтиже де болуы тиіс. Бұл шарттар осы жұмыста тексерілген қайта жазуды сипаттайды; нақты сұраудың жарамдылығын оңтайландырғыш әлі шешеді. Қайта жазу өшірулі және қосулы кезде **сол негізгі кесте SQL** салыстырыңыз. Екі орындау бір жауап береді, бірақ Query Profile әртүрлі орындалу сканерлеулерін көрсетуі тиіс: өшіруліде бастапқы кестелер, қосулыда `m08_async_category_metrics`.

In [ ]:
eligible_query = """
SELECT p.category, COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category
ORDER BY p.category
"""
without_rewrite = lab.capture_query(eligible_query,
    settings={"enable_materialized_view_rewrite": "false"})
without_rewrite.show_result("Rewrite disabled: category totals")
without_rewrite.show_profile_scan_rows("Rewrite disabled: source-table scan work")
with_rewrite = lab.capture_query(eligible_query,
    settings={"enable_materialized_view_rewrite": "true"})
with_rewrite.show_result("Rewrite enabled: category totals")
with_rewrite.show_profile_scan_rows("Rewrite enabled: materialized-view scan work")
lab.assert_same_rows(without_rewrite.rows, with_rewrite.rows)
lab.assert_scan(without_rewrite, "m08_events")
lab.assert_scan(without_rewrite, "m08_products")
lab.assert_scan(with_rewrite, "m08_async_category_metrics")
card("Complete category totals match; the enabled plan selects the asynchronous view.",
     "ok", "Eligible rewrite checked");


<a id="unit-u.2604e6cf75d4486687e9f844be42f219"></a>

**Нәтижені қалай оқу керек:**

- **Rewrite disabled: category totals** — бастапқы кестелерден есептелген жауап.
- **Rewrite enabled: category totals** — Doris асинхронды материалданған көріністі қайта пайдалана алғанда қайтарылған жауап. Екі көрсетілген кестеде сол тоғыз жол болуы тиіс және толық жолдар жиынының тексеруі әр мәнді салыстырады.
- Қайта жазу өшіруліде Query Profile `m08_events(m08_events)` және `m08_products(m08_products)` орындалу сканерлеулерін анықтайды.
- Қайта жазу қосулыда Query Profile `m08_async_category_metrics(m08_async_category_metrics)` анықтайды.
- `ScanRows` және `RowsProduced` — сол сканерлеулердің нақты орындау санағыштары. Толық өңделмеген Profile жазбалары қосымша қарауға жиналған қалады.

Бұл код қорабында `EXPLAIN` нәтижесі көрсетілмейді. Жоспар күтілімдері `assert_scan()` арқылы үнсіз тексеріледі. Санағыштар осы тексерілген орындауды сипаттайды; жалпы жеделдеу қатынасы емес.


<a id="unit-u.276f1459ba054a0387aff243c8268b88"></a>

<a id="ask-for-a-measure-the-view-did-not-preserve"></a>

### Көрініс сақтамаған метриканы сұрау

Негізгі кесте сұрауы сол Join, сүзгі және топтау өлшемдері бар екен деп бұл көріністі пайдалана алмайды. Сұралған метрикалар да көріністе сақталған күйден шығарылуы тиіс.

Енді санат бойынша бірегей сатып алушыларды сұраңыз. `m08_async_category_metrics` оқиға сандары мен түсім қосындыларын сақтайды, бірақ `user_id` мәндерін немесе біріктірілетін бірегей сатып алушы күйін сақтамайды. Сол метрикалар оқиғаларды қанша әртүрлі сатып алушы жасағанын қалпына келтіре алмайды. Әр аймаққа бөлек сақталған бірегей сан да жалпы жағдайда аймақтар арасында қосылмайды, өйткені бір сатып алушы бірнеше аймақта болуы мүмкін.

Бұл сұрау үшін Doris `m08_async_category_metrics` ішінен жауап шығара алмайды. Сондықтан қайта жазуды қосу бастапқы кесте жоспарын орнында қалдырады және Query Profile `m08_events(m08_events)` пен `m08_products(m08_products)` орындалу сканерлеулерін әлі анықтайды.

In [ ]:
ineligible_query = """
SELECT p.category, COUNT(DISTINCT e.user_id) AS distinct_buyers
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category
ORDER BY p.category
"""
ineligible_baseline = lab.capture_query(ineligible_query,
    settings={"enable_materialized_view_rewrite": "false"})
ineligible_baseline.show_result("Rewrite disabled: distinct buyers")
ineligible_baseline.show_profile_scan_rows("Rewrite disabled: source-table scan work")
ineligible = lab.capture_query(ineligible_query,
    settings={"enable_materialized_view_rewrite": "true"})
ineligible.show_result("Rewrite enabled: distinct buyers")
ineligible.show_profile_scan_rows("Rewrite enabled: source-table scan work")
lab.assert_same_rows(ineligible_baseline.rows, ineligible.rows)
lab.assert_scan(ineligible, "m08_events")
lab.assert_scan(ineligible, "m08_products")
card("Distinct-buyer results match; the selected scans still read the source tables.",
     "ok", "Missing-measure boundary checked");


<a id="unit-u.9ed864f077f7409fa355520554652f01"></a>

**Нәтижені қалай оқу керек:**

- Қайта жазу өшірулі және қосулы нәтиже кестелері бірдей тоғыз бірегей сатып алушы жиынтығын қамтуы тиіс. Тексеру әр қайтарылған мәнді салыстырады.
- Екі Query Profile де `m08_events(m08_events)` және `m08_products(m08_products)` анықтайды. Қайта жазу қосулы орындау `m08_async_category_metrics` атамайды.
- Материалданған көрініс оқиға сандары мен түсім қосындыларын сақтайды, бірақ `user_id` немесе біріктірілетін бірегей сатып алушы күйін сақтамайды, сондықтан Doris бастапқы егжей-тегжейлі деректерді оқуы тиіс.

Бұл код қорабында `EXPLAIN` нәтижесі көрсетілмейді. Жоспар күтілімдері `assert_scan()` арқылы үнсіз тексеріледі. Бұл — барлық бірегей санау сұрауы материалданған көрініспен жеделдетуге жарамсыз деген дәлел емес, осы анықтаманың шектеуі.


<a id="unit-u.ecdc5b3bf1ea43d58d517951fb0afb45"></a>

<a id="6-isolate-an-inverted-index-for-product-investigation"></a>

## 6. Өнімді зерттеу үшін инверттелген индексті оқшаулау

Қолдау талдаушысына алдын ала есептелген агрегат емес, сирек бір өнімнің оқиғалары қажет. Бағандары, Duplicate Key реті, таратуы және кіріс жолдары бірдей екі оқиға кестесін жасаңыз. Тек біреуі `product_id` бойынша Inverted Index жариялайды.

`product_id` сандық, сондықтан бұл мәтін токендеушісі жоқ теңдік индексі. Ол толық мәтін `MATCH` мысалы да, Join Runtime Filter де емес. Предикат сұрыптаудың бастапқы бағанын немесе `user_id` тарату бағанын шектемейді. Индексті дерек жүктеуден бұрын жариялау бөлек тарихи индекс құру қадамын болдырмайды.

In [ ]:
for table in ("m08_events_indexed", "m08_events_unindexed"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")

lab.execute("""
CREATE TABLE m08_events_unindexed (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
lab.execute("""
CREATE TABLE m08_events_indexed (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    INDEX m08_product_idx(product_id) USING INVERTED
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
for table in ("m08_events_unindexed", "m08_events_indexed"):
    lab.insert(f"INSERT INTO {table} SELECT * FROM m08_events", title=f"Load {table}")
    lab.execute(f"ANALYZE TABLE {table} WITH SYNC")

rare_product = lab.sql("""
SELECT product_id, COUNT(*) AS matching_events
FROM m08_events_unindexed
GROUP BY product_id
HAVING SUM(revenue) > 0
ORDER BY matching_events, product_id
LIMIT 1
""", title="Product selected for the indexed lookup")
rare_product_id = int(rare_product.iloc[0]["product_id"]);

<a id="unit-u.82038699ab414e5394aafe755c05e057"></a>

**Күтілетін нәтиже:** екі кесте сол ағымдағы `m08_events` көшірмесінен бірдей `INSERT ... SELECT` арқылы жүктеледі. Көрінетін DDL тек `m08_events_indexed` `m08_product_idx` жариялағандықтан ерекшеленеді; екеуінде де материалданған көрініс жоқ.

Бір жолдық нәтиже екі іздеу сұрауы қолданатын өнімді және сәйкес оқиға санын анықтайды. Оң түсімі бар ең сирек өнімді таңдау дерекке тәуелді `product_id` мәнін қатаң жазбай, іріктемелі предикат береді. Көшірілген деректерге 3-бөлімде қосылған өзімізге тиесілі бір сатып алу кіреді. Дербес жүктелген екі кестенің физикалық беттері, Compaction күйі және кэште тұруы міндетті түрде бірдей емес.

In [ ]:
lookup_template = """
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM {table}
WHERE product_id = {product_id}
ORDER BY event_time, event_id, user_id
"""
unindexed_lookup = lab.capture_query(lookup_template.format(
    table="m08_events_unindexed", product_id=rare_product_id))
indexed_lookup = lab.capture_query(lookup_template.format(
    table="m08_events_indexed", product_id=rare_product_id))
lab.show_lookup_comparison({"Without index": unindexed_lookup, "With index": indexed_lookup})
unindexed_lookup.show("Product lookup without the secondary index", detail=True)
indexed_lookup.show("Product lookup with the inverted index", detail=True)
lab.assert_same_rows(unindexed_lookup.rows, indexed_lookup.rows)
assert len(indexed_lookup.rows) == int(rare_product.iloc[0]["matching_events"])
lab.assert_scan(unindexed_lookup, "m08_events_unindexed")
lab.assert_scan(indexed_lookup, "m08_events_indexed")
card("Every returned event matches. Read the per-task index counters before attributing scan savings.",
     "ok", "Selective lookup checked");

<a id="unit-u.14b18ea65e8644eaa344a65d178e0a8b"></a>

**Күтілетін нәтиже:** екі сұрау сол оқиғаларды қайтарады. **Lookup comparison** бөлімінен бастаңыз: ол Profile ID, сұрау нәтиже саны және өңделмеген `ScanRows` пен `RowsInvertedIndexFiltered` қатар қояды. Әр жол — бір DetailProfile Scan тапсырмасы; бірнеше тапсырмасы бар сұрау нәтиже санын қайталайды, сондықтан ол бағанды қоспаңыз. Жоқ санағыш `not shown` деп, байқалған нөл `0` болып қалады. `EXPLAIN` Scan белгісі кестенің негізгі индексін әлі атайды; өзі қосымша индекс сүзуін дәлелдемейді. Әр сұрауға **DetailProfile scan counters** ашып, әр санағышпен Fragment, host, PipelineTask және Scan сәйкестігін бірге сақтаңыз: DetailProfile әр PipelineTask, бір BE-дегі сканерлеудің бір параллель тапсырмасын бөлек тізеді.

| Дәлел | Түсіндіру |
|---|---|
| `RowsInvertedIndexFiltered` | Сол Scan тапсырмасында инверттелген индекс жолымен сүзілген жолдар; оң мәндер нақты индекс сүзуін қолдайды. |
| `InvertedIndexQueryTime`, `InvertedIndexFilterTime` | Бастапқы уақыт бірліктерімен индекс жолындағы тіркелген жұмыс. Бұлар бүкіл сұраудың толық кідірісі емес. |
| `InvertedIndexQueryCacheHit` / `Miss` | Индекс кэш оқиғалары; түсіру көмекшісі бұл кэшті өшірмеген. |
| `RowsStatsFiltered`, `RowsKeyRangeFiltered` | Жұмысты азайта алатын басқа іріктеу жолдары. Өткізілген әр жолды жаңа индекске телімеңіз. |
| `ScanRows` | Аталған Scan астында сақтау орнынан оқылған жолдар, бет кэшін қоса. |
| `RowsRead`, `RowsProduced` | Өз ауқымдарындағы scanner/шығыс жол санағыштары; соңғы SQL нәтижесі дұрыстық тексеруі болып қалады. |
| `ScanBytes` | Тіркелген сканерлеу байттары, болдырылған физикалық диск тасымалының өлшемі емес. |

Сыналған жиында индекстелген орындау оң `RowsInvertedIndexFiltered` мәндерін және аз `ScanRows` тіркейді. Нақты сан мен уақыт сақтау орналасуы және кэштермен өзгеруі мүмкін. Санағыш жоқ болса, ол нөл емес, **көрсетілмеген**; дәлелді тексеру үшін толық өңделмеген Profile пайдаланыңыз. Біріктірілген қорытынды мен әр тапсырманың егжей-тегжейі қабаттасқан жұмысты сипаттайды — оларды ешқашан қоспаңыз.

Іріктемелі индекс қосымша сақтау мен жазу/қолдау жұмысының орнына іздеу жұмысын азайтады. Бұл салыстыру өндірістік жүктемеде сол таңдауды сандық бағаламайды және тұрақты жеделдеуді дәлелдемейді. Осы бір түйінді жұмыстан көп BE өнімділігі туралы қорытынды шықпайды.

<a id="unit-u.d1406c9db94140aa9e654c8a45307b1f"></a>

<a id="7-read-the-statistics-behind-the-chosen-plan"></a>

## 7. Таңдалған жоспардың артындағы статистиканы оқу

Бұл жұмыстағы әр жоспарды шығынға негізделген оңтайландырғыш таңдады; ол әр оператор қанша жол өңдейтінін **статистикадан** болжайды: кесте жол саны және әр баған үшін бірегей мән саны (`ndv`), `NULL` саны, минимум мен максимум. Статистика `ANALYZE` арқылы жиналады; бұл сұрау бақылауы емес, жоспарлау болжамдары.

Келесі ұяшық санат есебі оқитын екі кестеге жаңа статистика жинайды, оператор жинау аяқталғанда қайтуы үшін `WITH SYNC` қолданып, Doris тіркегенін көрсетеді. Мұндағыдай тек ағымдағы шешім кестелеріне статистика жинаңыз, тұтас ортақ дерекқорға емес.

In [ ]:
lab.execute("ANALYZE TABLE m08_events WITH SYNC")
lab.execute("ANALYZE TABLE m08_products WITH SYNC")

statistics_columns = ["column_name", "count", "ndv", "num_null", "min", "max", "method"]
lab.sql(
    "SHOW COLUMN STATS m08_events (event_type, product_id)",
    title="Column statistics: m08_events",
    columns=statistics_columns,
)
lab.sql(
    "SHOW COLUMN STATS m08_products (product_id)",
    title="Column statistics: m08_products",
    columns=statistics_columns,
);

<a id="unit-u.3ff68e8fc7bc45d29580e28d36e44b9e"></a>

**Күтілетін нәтиже**

| Кесте және баған | `count` | `ndv` | `num_null` | `min` / `max` |
|---|---:|---:|---:|---|
| `m08_events.event_type` | 75260 | 2 | 0 | `'cart'` / `'purchase'` |
| `m08_events.product_id` | 75260 | шамамен 12,400 | 0 | 1001606 / 100145255 |
| `m08_products.product_id` | 12339 | шамамен 12,400 | 0 | 1001606 / 100145255 |

`method` — `FULL`: `ANALYZE TABLE` әр жолды оқыды. `count` — көшірілген 75,259 оқиға және 3-бөлімде қосылған сатып алу. Күнде тек себет пен сатып алу оқиғалары бар, сондықтан `event_type` екі бірегей мәнге ие. `ndv` өзі де болжам: сыналған орындауда екі `product_id` бағанында 12,412 болды, ал кестелерде 12,340 және 12,339 әртүрлі өнім бар.

<a id="unit-u.f42c40f01be54cac82956055a1aec829"></a>

Енді асинхронды материалданған көріністің орнына екі кестені оқу үшін мөлдір қайта жазуды өшіріп, оңтайландырғыш 4-бөлімнің санат есебіне таңдаған жоспарды оқыңыз. Ұяшық оператор жолдарын, сканерленетін кестелерді, Join және әр `cardinality=` болжамын сақтайды; толық жоспар жиналған панельде. Содан кейін Join орындап, одан нақты шығатын жолдарды санайды.

In [ ]:
import re
from doris_course.ui import show_log

with lab.session_settings({"enable_materialized_view_rewrite": "false"}):
    plan = "\n".join(lab.query("EXPLAIN " + category_query).iloc[:, 0])
plan_outline = [line for line in plan.splitlines()
                if re.search(r"\d+:V|TABLE:|join op:|cardinality=", line)]
show_log("Chosen plan of the category report: operators, scans, and estimated rows",
         "\n".join(plan_outline), opened=True)
show_log("Complete plan of the category report", plan)

lab.sql("""
SELECT COUNT(*) AS purchases_with_a_product
FROM m08_events e
JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
""", title="Rows that actually leave the Join");

<a id="unit-u.fad00c105faa469cb842ef6fa5f8d74b"></a>

**Күтілетін нәтиже**

- Жоспар `event_type = 'purchase'` предикатымен `m08_events` және `m08_products` оқып, `join op: INNER JOIN(BROADCAST)` арқылы біріктіреді: шамамен 12,340 жолдық өнім жағы оқиға сканерлеуіне жіберіліп, hash кестесіне айналады.
- Join болжамы шамамен `cardinality=37,400` (сыналған орындауда 37,408). `event_type` үшін `ndv = 2` болып, әр мән жиілігі туралы ақпарат болмағандықтан, оңтайландырғыш 75,260 жолдың жартысын сатып алу деп болжады.
- `purchases_with_a_product` — **17,798**: жолдардың 18,299-ы, шамамен 24%, сатып алу; ал 1005115 өнімінің 501 сатып алуына өнім жолы жоқ.

Толық жоспар `planned with unknown column statistics` деп аяқталады. Ескерту жоспар `m08_products` оқыған сайын пайда болады: Unique Key кестесінде жойылған жолдарды белгілейтін жасырын баған бар, Scan оны сүзеді (`__DORIS_DELETE_SIGN__ = 0`), ал `ANALYZE` ол бағанға статистика жинамайды.

Болжам нақты саннан шамамен екі есе, бірақ мұнда нашар жоспарға әкелмеді: өнім жағы бәрібір шағын, сондықтан Broadcast арзанырақ Join болып қалады. Болжам оңтайландырғышты күткен құрылымнан немесе Join ретінен бұрса, сабақтың 8.7-бөліміндегідей алдымен сұрау пішінін, құрылым күйін және статистиканы тексеріңіз. 6-модульдегі `[shuffle]` сияқты hint бақыланатын салыстыруға басқа жоспарды мәжбүрлей алады; статистиканы түзетпейді.

<a id="unit-u.08a06904569d42bca0c16b64050348dd"></a>

<a id="8-practice-ask-the-materialized-view-for-a-new-grain"></a>

## 8. Жаттығу: материалданған көріністен жаңа түйіршіктілікті сұрау

`m08_async_category_metrics` сатып алуларды `(category, region)` бойынша сақтайды. Ол сақтамайтын түйіршіктілікте есеп жазыңыз: сол `m08_events` және `m08_products` Join амалы бойынша **тек аймақпен** сатып алу саны мен түсім.

Орындамас бұрын екі нәрсені болжаңыз:

1. Мөлдір қайта жазу қосылғанда, Doris есепке `m08_async_category_metrics` арқылы жауап бере ала ма, әлде екі кестені оқуы тиіс пе?
2. Оның жиынтықтары тек `m08_events` оқыған 2-бөлім аймақ есебіне тең бе? Онда `region_01` қазір 2,320: 2-бөлімнің 2,319 мәні және 3-бөлім қосқан сатып алу.

Содан кейін есепті `lab.capture_query(..., settings={"enable_materialized_view_rewrite": "true"})` арқылы орындап, нәтижесі мен таңдалған Scan көрсетіңіз және оқиға сандарын қосыңыз.

In [ ]:
# Your code here

<a id="unit-u.7092156614ae464eaa024bca1be8d249"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
region_join_query = """
SELECT e.region, COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY e.region
ORDER BY e.region
"""
region_join = lab.capture_query(region_join_query,
    settings={"enable_materialized_view_rewrite": "true"})
region_join.show_result("Purchases with a product, by region")
region_join.show_scan_plan("Selected scan with rewrite enabled")
lab.assert_scan(region_join, "m08_async_category_metrics")
assert int(region_join.rows["event_count"].astype(int).sum()) == 17_798;

<a id="unit-u.3f5f0bd0c1db4645871ba9ea173bc27c"></a>

**Күтілетін нәтиже**

1. **Иә.** Таңдалған Scan — `m08_async_category_metrics(m08_async_category_metrics)` және тексерулер үнсіз өтеді. Есеп көріністің Join, сүзгі және метрикаларын сақтап, тек топтаудан `category` алып тастайды; сандар мен қосындылар қосылатындықтан, Doris 5-бөлімнің санат жиынтықтарындағыдай әр аймақтың көріністегі санат жолдарын қосады.
2. **Жоқ.** Мұнда `region_01` **2,251** сатып алуға және 672,822.82 түсімге ие, тікелей есепте 2,320 (2-бөлімдегі 2,319 және 3-бөлімде қосылған сатып алу), ал сегіз санның қосындысы 18,299 емес, **17,798**. Inner Join өнім жолы жоқ 1005115 өнімінің 501 сатып алуын алып тастайды. Көрініс Join есебіне дұрыс жауап береді; 1-бөлім ескерткендей, бұл тікелей есептен басқа жиын.

<a id="unit-u.29c1b4f564694b8f95163ff266405397"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

Жұмысты аяқтап, контейнердің CPU мен жадын босатқыңыз келгенде осы қосымша ұяшықты орындаңыз. Ол Doris үдерістерін тоқтатады, бірақ контейнерді, образды, FE метадерек томын және BE сақтау томын сақтайды. Сынақ ортасын пайдалануды жалғастырғыңыз келсе, өткізіп жіберіңіз.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");


<a id="unit-u.1cab63f4253949cc9bed809f9bfc20f5"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды. Бастапқы кестелер мен барлық `m08_*` нысандары атаулы томдарда қалады.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Сол 8-зертханалық жұмыс нысандарымен жалғастырғыңыз келгенде осы ұяшықты орындаңыз. Ол бар контейнерді іске қосып, Doris дұрыс жұмыс істейтін күйге жеткенше күтеді, FE түйініне қайта қосылады және сақталған оқиға кестесін тексереді.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container()

lab.connect()
lab.execute("USE doris_course")
lab.sql("""
SELECT
    COUNT(*) AS recovered_rows,
    SUM(event_id = -8001) AS owned_purchase_rows
FROM m08_events
""", title="Recovered Lab 8 event table");


<a id="unit-u.424c679ba1624d59abfef2a89a100192"></a>

**Күтілетін нәтиже:** контейнер `healthy` күйіне оралады. Толық жұмыс орындалғаннан кейін `recovered_rows` — **75,260**, ал `owned_purchase_rows` — **1**. Бұл бастапқы көшірме мен 3-бөлімдегі бір идемпотентті сатып алудың тоқтатудан кейін сақталғанын растайды.



<a id="unit-u.85db3b2f88fb4d92b2484e39ff29c2fe"></a>

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Қалыпты көрініс арқылы бір сатып алу анықтамасын қайта пайдаландыңыз, синхронды материалданған индекстің жазудан кейін жарамды жиынтықты қолдайтынын байқадыңыз, асинхронды материалданған көріністің жаңарту шекарасын басқардыңыз, сұрауды қайта жазу оның сақталған түйіршіктілігі мен метрикаларын қашан қайта пайдалана алатынын тексердіңіз, инверттелген индекс іздеуін негізгі кесте сканерлеуінен ажырату үшін Query Profile дәлелін қолдандыңыз және Join туралы оңтайландырғыш болжамын одан нақты шыққан жолдармен салыстырдыңыз. Модульге тиесілі нысандар `doris_course` ішінде қалады және жоғарыдағы тоқтату мен қайта іске қосу ұяшықтары оларды кейін тексеруге сақтайды.

Ресми анықтамалар: [Синхронды материалданған көріністер](https://doris.apache.org/docs/4.x/query-acceleration/materialized-view/sync-materialized-view/) · [Асинхронды материалданған көріністер](https://doris.apache.org/docs/4.x/query-acceleration/materialized-view/async-materialized-view/functions-and-demands/) · [Инверттелген индекс](https://doris.apache.org/docs/4.x/key-features/inverted-index/) · [Query Profile](https://doris.apache.org/docs/4.x/query-acceleration/query-profile/)

<a id="unit-u.e096698bf18d404d97af85f2b7b7e2f2"></a>

<a id="next"></a>

## Келесі қадам

[8-тест: Apache Doris жүйесінде сұрауларды жеделдету](quiz08_acceleration_choices.ipynb) ашыңыз, содан кейін [9-модуль: Apache Doris жүйесінде шардтау, репликация және үлестірілген орындау](../module09-sharding-replication/course.md) бөліміне өтіңіз.